In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO     
from PIL import ImageFont, ImageDraw, Image   
import os
from datetime import datetime    
from tkinter import filedialog
import tkinter as tk          # 윈도우 탐색기를 띄워 영상 파일을 선택하기 위해 사용
import easyocr                # 이미지 속 글자를 텍스트로 변환하는 OCR엔진
import re

In [ ]:
# 1. 모델 로드 (YOLO 번호판 탐지만 사용)
model_01 = YOLO(r'D:\미니프로젝트\plate_detection\01_detection\runs\korea_plate_11small\weights\best.pt')

# 2. EasyOCR 초기화
reader = easyocr.Reader(['ko', 'en'], gpu=True)

# 저장 폴더 설정
save_path = "captured_plates"
if not os.path.exists(save_path):
    os.makedirs(save_path)

font_path = "C:/Windows/Fonts/malgun.ttf" #맑은 고딕체

def draw_ko_text(img, text, position, color=(0, 255, 0), font_size=30):
    img_pil = Image.fromarray(img) # opencv 형식(numpy array)의 이미지를 PIL형식의 이미지로 변환 이유는 opencv는 한글 폰트(ttf)를 직접 다루지 못하지만, PIL은 이를 지원하기 때문에 변환
    draw = ImageDraw.Draw(img_pil) # 변환된 PIL 이미지 위에 글자를 그릴수 있는 도화지 객체
    current_font = ImageFont.truetype(font_path, font_size) 
    draw.text(position, text, font=current_font, fill=color)
    return np.array(img_pil)

# 영상 파일 선택
def select_video_file():
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    file_path = filedialog.askopenfilename(
        title="분석할 영상을 선택하세요",
        filetypes=[("Video files", "*.mp4 *.avi *.mkv *.mov"), ("All files", "*.*")]
    )
    root.destroy()
    return file_path

def process_video(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    # 우측 정보창 초기화 (720x400)
    info_panel = np.zeros((720, 400, 3), dtype=np.uint8)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        cur_frame = cap.get(cv2.CAP_PROP_POS_FRAMES)
        total_sec = int(cur_frame / fps)
        timestamp = f"{total_sec // 60:02d}:{total_sec % 60:02d}"

        frame = cv2.resize(frame, (1280, 720))
        results_01 = model_01(frame, conf=0.3, verbose=False)[0]

        # 이번 프레임에서 발견된 모든 번호판 정보를 담을 리스트
        detected_list = []

        for box in results_01.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            plate_roi = frame[y1:y2, x1:x2]
            if plate_roi.size == 0: continue

            # --- [EasyOCR 단계] ---
            gray_plate = cv2.cvtColor(plate_roi, cv2.COLOR_BGR2GRAY)
            # 대비를 높이는 선명화 작업 추가
            gray_plate = cv2.threshold(gray_plate, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
            # 이미지 크기 확대
            gray_plate = cv2.resize(gray_plate, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

            ocr_results = reader.readtext(gray_plate, detail=0) 
            raw_txt = "".join(ocr_results)
            txt = re.sub(r'[^0-9가-힣]', '', raw_txt) 
            
            if len(txt) >= 4:
                detected_list.append({'text': txt, 'roi': plate_roi})
                
                # 메인 화면 표시
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                frame = draw_ko_text(frame, txt, (x1, y1 - 40))

        # 화면 상단 안내
        frame = draw_ko_text(frame, f"TIME: {timestamp} | 'C': Capture All", (20, 20), (255, 255, 255), 20)
        
        cv2.imshow('Main Analysis Window', frame)
        cv2.imshow('Captured Info Panel', info_panel)

        key = cv2.waitKey(1) & 0xFF
        
        # --- [전략 C: 리스트형 정보창 업데이트] ---
        if key == ord('c') and detected_list:
            info_panel = np.zeros((720, 400, 3), dtype=np.uint8)
            info_panel = draw_ko_text(info_panel, f"검출 결과 ({len(detected_list)}대)", (20, 20), (255, 255, 0), 25)
            
            for i, item in enumerate(detected_list[:5]): # 최대 5대 표시
                y_offset = 70 + (i * 130)
                
                # 1. 번호판 이미지 리사이즈 및 삽입
                roi = item['roi']
                rh, rw = roi.shape[:2]
                # 리스트에 넣기 위해 작게 리사이즈 (가로 150)
                resized_roi = cv2.resize(roi, (150, int(rh * (150/rw))))
                ih, iw = resized_roi.shape[:2]
                info_panel[y_offset:y_offset+ih, 20:20+iw] = resized_roi
                
                # 2. 차량 종류 판별
                v_type = "일반 차량"
                if any(h in item['text'] for h in ['하', '허', '호']): v_type = "렌터카"
                elif any(s in item['text'] for s in ['바', '사', '자', '배']): v_type = "영업용"
                
                # 3. 우측 텍스트 정보 기입 (좌표를 i에 따라 계산)
                info_panel = draw_ko_text(info_panel, f"번호: {item['text']}", (180, y_offset), (0, 255, 0), 22)
                info_panel = draw_ko_text(info_panel, f"시간: {timestamp}", (180, y_offset + 30), (255, 255, 255), 18)
                info_panel = draw_ko_text(info_panel, f"정보: {v_type}", (180, y_offset + 55), (0, 255, 255), 20)
                info_panel = draw_ko_text(info_panel, f"날짜: {datetime.now().strftime('%m-%d')}", (180, y_offset + 80), (150, 150, 150), 16)
                
                # 4. 구분선
                cv2.line(info_panel, (20, y_offset + 115), (380, y_offset + 115), (60, 60, 60), 1)

                # 5. 모든 파일 자동 저장
                save_fn = f"{save_path}/{v_type}_{item['text']}_{datetime.now().strftime('%H%M%S')}_{i}.jpg"
                cv2.imwrite(save_fn, item['roi'])
            
            print(f"총 {len(detected_list)}대의 번호판 정보가 업데이트 및 저장되었습니다.")

        elif key == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    video_target = select_video_file()
    if video_target:
        process_video(video_target)

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
